# Image Classification

In [ ]:
from datasets import load_dataset
dataset = load_dataset("nlphuji/flickr30k")
image = dataset['test']["image"]

from transformers import pipeline
pipe = pipeline("image-classification", "google/mobilenet_v2_1.0_224")
pred = pipe(image)
print("Predicted class:", pred['label'])


# Object Detection

In [ ]:
pipe = pipeline("object-detection", "facebook/detr-resnet-50", revision="no_timm")
outputs = pipe(image, threshold=0.95)
for obj in outputs:
    box = obj['box']
    print(f"Detected {obj['label']} with confidence {obj['score']:.2f} at ({box['xmin']}, {box['ymin']}) to ({box['xmax']}, {box['ymax']})")


# Segmentation

In [ ]:
pipe = pipeline("image-segmentation", model="briaai/RMBG-1.4", trust_remote_code=True)
outputs = pipe(image)


# Fine-Tuning Vision

#  تجهيز الفئات (Label Mapping)

In [ ]:
labels = data_splits["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label


# تحميل النموذج وإعادة تشكيل الطبقة الأخيرة

In [ ]:
from transformers import AutoModelForImageClassification
checkpoint = "google/mobilenet_v2_1.0_224"
model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)


# معالجة الصور رياضياً (Data Preprocessing)

In [ ]:
from transformers import AutoImageProcessor
image_processor = AutoImageProcessor.from_pretrained(checkpoint)
from torchvision.transforms import Compose, Normalize, ToTensor

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
transform = Compose([ToTensor(), normalize])

def transforms(examples):
    examples["pixel_values"] = [transform(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples
dataset = dataset.with_transform(transforms)


#  إعدادات حلقة التدريب (Trainer)

In [ ]:
from transformers import TrainingArguments, Trainer, DefaultDataCollator
training_args = TrainingArguments(
    output_dir="dataset_finetune",
    learning_rate=6e-5,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    push_to_hub=False
)
data_collator = DefaultDataCollator()
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=image_processor,
    data_collator=data_collator
)
trainer.train()


# التعرف على الصوت (Whisper ASR)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))
sample = dataset["audio"]

input_preprocessed = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt")
predicted_ids = model.generate(input_preprocessed.input_features)
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)
print(transcription)


# توليد الصوت والضبط الدقيق (SpeechT5 TTS)

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier
speaker_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb")

speaker_embeddings = speaker_model.encode_batch(torch.tensor(dataset["audio"]["array"]))
speaker_embeddings = torch.nn.functional.normalize(speaker_embeddings, dim=2).unsqueeze(0)


# إعدادات الضبط الدقيق للصوت (Seq2Seq Fine-tuning)

In [ ]:
from transformers import Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    label_names=["labels"],
)


# Inference with Vocoder

In [ ]:
inputs = processor(text=text, return_tensors="pt")
speech = model.generate_speech(
    inputs["input_ids"],
    speaker_embedding,
    vocoder=vocoder
)
